In [ ]:
#import packages
import numpy as np
import netCDF4 as nc
import xarray as xr
import calendar
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import mapping
import geopandas as gpd
from sklearn.linear_model import TheilSenRegressor
import pymannkendall as mk
import gc
import os

#add path and prepared data
path1=' '
in_path=path1+" "

In [ ]:
som_grids = { 'grid3x4': (3, 4),}   #  'grid3x3': (3, 3),'grid4x4': (4, 4),
#grid_name='grid3x4'
#grid_size=(3, 4)

In [ ]:
gph_anom=xr.open_dataarray("uyr_gph_anom_1979-2022.nc")
tp_anom=xr.open_dataarray("uyr_tp_anom_1979-2022.nc") 
temp_anom=xr.open_dataarray("uyr_temp_anom_1979-2022.nc") 
print(gph_anom)
time=gph_anom.time.values
lat=gph_anom.latitude
lon=gph_anom.longitude
lons, lats = np.meshgrid(lon, lat)

shp = gpd.read_file("D:/Data/uyr/shp/Upper_Yangtze_River.shp")

def clip_arr(arr):
    arr.rio.write_crs("epsg:4326", inplace=True)
    arr.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=True)
    clip_data=arr.rio.clip(shp.geometry, shp.crs, drop=False, all_touched=True)
    return clip_data

clip_da=clip_arr(gph_anom)

rigional_combo=[]
combo=[] 
da=clip_da[0,:,:]
for j in range(gph_anom.shape[1]):
    for k in range(gph_anom.shape[2]):
        combo.append((j,k))
        if not np.isnan(da[j, k].values):
            rigional_combo.append((j, k))           
#print(rigional_combo) #len=1674

indices = []
for item in rigional_combo:
    idx = combo.index(item)
    indices.append(idx)
print(indices)   #len=1674

<xarray.DataArray 'gph_anom' (time: 16060, latitude: 49, longitude: 89)> Size: 560MB
[70037660 values with dtype=float64]
Coordinates:
  * longitude  (longitude) float32 356B 90.0 90.25 90.5 ... 111.5 111.8 112.0
  * latitude   (latitude) float32 196B 36.0 35.75 35.5 35.25 ... 24.5 24.25 24.0
  * time       (time) datetime64[ns] 128kB 1979-01-01 1979-01-02 ... 2022-12-31
[104, 105, 106, 182, 183, 184, 185, 186, 187, 188, 189, 190, 192, 193, 194, 195, 196, 197, 198, 199, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 536, 537, 538, 539, 540, 541, 542, 543, 544, 545, 546, 547, 548, 549, 550, 551, 552, 553, 554, 555, 556, 557, 558, 559, 560, 591, 593, 594, 595, 596, 597, 598, 599, 600, 625, 626, 627, 628, 629, 630, 631, 

In [ ]:
#som node link total
vars=['tp','temp']
from matplotlib.colors import TwoSlopeNorm
import cartopy.crs as ccrs

shp = gpd.read_file("Upper_Yangtze_River.shp")

def clip_arr(data):
    data.rio.write_crs("epsg:4326", inplace=True)
    data.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=True)
    clip_data=data.rio.clip(shp.geometry, shp.crs, drop=False, all_touched=True)
    return clip_data

annotate_kwargs1 = {
    'xycoords': 'axes fraction',
    'horizontalalignment': 'left',
    'fontsize': 7.5,
    'color': 'black',
    'bbox': dict(facecolor='lightgrey', edgecolor='None', alpha=0.8, pad=0.5)
}

annotate_kwargs2 = {
    'xycoords': 'axes fraction',
    'horizontalalignment': 'right',
    'fontsize': 9,
    'color': 'black',
    'bbox': dict(facecolor='lightgrey', edgecolor='None', alpha=0.8, pad=0.5)
}


som_grids = {'grid3x4': (3, 4),}   #  'grid4x4': (4, 4),  'grid3x3': (3, 3),

season_name=['total']  #,'Spring','Summer','Autumn','Winter'

years = np.arange(1979, 2023)

for grid_name, grid_size in som_grids.items():

    input_combo=[]  
    for j in range(grid_size[0]):
        for k in range(grid_size[1]):                 
                input_combo.append((j,k))   
    #print(len(input_combo))   

    feature=pd.read_excel(in_path+"feature/gph_node_"+grid_name+"_total.xlsx")
    
    classif=pd.read_csv(in_path+"som_results/gph_grid3x4_total_classif.csv")  #pd.DataFrame
    #print(classif.shape)  (len(time), 1)
    #classif.rename(columns={'x': 'node'}, inplace=True)
    classif["time"]=time.reshape((len(time),1))
    #classif['Time'] = pd.to_datetime(df['Time'])  
    classif["num"]=np.arange(1, len(time)+1, 1)
    
    group = classif.groupby('node').agg({'time': list, 'num': list})
    #print(group)

    for var in vars:
        if var=='tp':
            data_anom=tp_anom
            cmap="RdBu"
            text1="mm"
            cbar_level=np.arange(-2,2+1,1)
            levels = np.arange(-2, 2.1, 0.1)
        else:
            data_anom=temp_anom
            cmap="RdBu_r"
            text1="℃"
            cbar_level=np.arange(-3,3+1,1)
            levels = np.arange(-3, 3.1, 0.15)

        fig, axs = plt.subplots(grid_size[0], grid_size[1], subplot_kw=dict(projection=ccrs.PlateCarree()),figsize=(12, 8))

        for n in range(len(input_combo)):
            (i, j) = input_combo[n]
            node_time=pd.to_datetime(group.loc[n+1, 'time'])
            #print(node_time)

            var_data = data_anom.sel(time=node_time).mean(dim='time')

            clip_data=clip_arr(var_data)
            var_mean=np.mean(clip_data).values

            ax=axs[i,j]
            ax.annotate(f'#{n+1}', xy=(0.98, 0.80), **annotate_kwargs2)
            ax.annotate(f'{var_mean:.4f}', xy=(0.01, 0.06), **annotate_kwargs1)

            mk_result_per_p=feature['mk_result_per_p'][n]
            slope_per=feature['slope_per'][n]

            if slope_per<0:
                color='blue'
            else:
                color='red'
            
            if mk_result_per_p<0.01:
                text='**'
            elif mk_result_per_p<0.05:
                text='*'
            else:
                text=''
                
            ax.annotate(f'{slope_per:.4f}', xy=(0.98, 0.04), xycoords='axes fraction',horizontalalignment='right',fontsize=6,color=color)
            ax.annotate(text, xy=(0.98, 0.14), xycoords='axes fraction',horizontalalignment='right',fontsize=9,color=color)

            #ax.coastlines(linewidth=0.2, alpha=1)
            ax.add_geometries(shp.geometry, crs=ccrs.PlateCarree(), facecolor='none', edgecolor='black', linewidth=0.2)
            
            ax.set_extent([90, 112, 36, 24], crs=ccrs.PlateCarree())
            cf = ax.contourf(lons, lats, clip_data, transform=ccrs.PlateCarree(), levels=levels, extend='both', cmap=cmap, norm = TwoSlopeNorm(vcenter=0))

            ax.set_title(f'{(len(node_time)/len(time)):.2%}',fontsize=6.8,pad=2)

        
            for spine in ax.spines.values():
                spine.set_linewidth(0.2)

        fig.subplots_adjust(wspace=0.05, hspace=0.05)         
        plt.subplots_adjust(left=0.3, right=0.7, top=0.7, bottom=0.35) 

        cbar = fig.colorbar(cf, ax=axs[:,:], orientation='horizontal', aspect=60, pad=0.03)
        cbar.set_ticks(cbar_level)
        cbar.outline.set_linewidth(0.2)
        cbar.ax.tick_params(labelsize=6, length=2)

        fig.text(0.69,0.3745,text1,size=6.5)

        fig.savefig(in_path+f"{var}_anom_link_grid3x4_total_clip.png",dpi=1000,bbox_inches='tight')
        plt.show()
        

In [ ]:
#load data. gph and water vapor in two dimensions
qu=xr.open_dataarray("iwv/qu_1979-2022.nc")
qv=xr.open_dataarray("iwv/qv_1979-2022.nc")
gph=xr.open_dataarray("gph/gph_500_1979-2022.nc")

In [ ]:
#gph-iwv-link-EA total
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import Normalize

shp = gpd.read_file("Upper_Yangtze_River.shp")

def clip_arr(arr):
    data=arr.rename({'latitude': 'lat', 'longitude': 'lon'})
    data.rio.write_crs("epsg:4326", inplace=True)
    data.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
    clip_data=data.rio.clip(shp.geometry, shp.crs, drop=False, all_touched=True)
    return clip_data

annotate_kwargs1 = {
    'xycoords': 'axes fraction',
    'horizontalalignment': 'left',
    'fontsize': 7.5,
    'color': 'black',
    'bbox': dict(facecolor='lightgrey', edgecolor='None', alpha=0.8, pad=0.5)
}

annotate_kwargs2 = {
    'xycoords': 'axes fraction',
    'horizontalalignment': 'right',
    'fontsize': 8,
    'color': 'black',
    'bbox': dict(facecolor='lightgrey', edgecolor='None', alpha=0.8, pad=0.5)
}


som_grids = {'grid3x4': (3, 4),}   #  'grid4x4': (4, 4),  'grid3x3': (3, 3),

season_name=['total']  #,'Spring','Summer','Autumn','Winter'

years = np.arange(1979, 2023)

interval=20

for grid_name, grid_size in som_grids.items():

    input_combo=[]  
    for j in range(grid_size[0]):
        for k in range(grid_size[1]):                 
                input_combo.append((j,k))   
    #print(len(input_combo))   

    feature=pd.read_excel(in_path+"feature/gph_node_"+grid_name+"_total.xlsx")
    
    season_classif = {}

    data=xr.open_dataarray("EastAsia_ERA5/gph/gph_anom_1979-2022.nc")

    #print(data)

    time=data.time.values
    lon=data.longitude.values
    lat=data.latitude.values

    lons, lats = np.meshgrid(lon, lat)

    lat1=lat[::interval]
    lon1=lon[::interval]

    lons1, lats1 = np.meshgrid(lon1, lat1)

    classif=pd.read_csv(in_path+"som_results/gph_grid3x4_total_classif.csv")  #pd.DataFrame
    #print(classif.shape)  (len(time), 1)
    #classif.rename(columns={'x': 'node'}, inplace=True)
    classif["time"]=time.reshape((len(time),1))
    #classif['Time'] = pd.to_datetime(df['Time'])  
    classif["num"]=np.arange(1, len(time)+1, 1)

    group = classif.groupby('node').agg({'time': list, 'num': list})

    fig, axs = plt.subplots(grid_size[0], grid_size[1], subplot_kw=dict(projection= ccrs.LambertConformal(central_longitude=105, standard_parallels=(0, 60))), figsize=(10, 7.5))
    for n in range(len(input_combo)):
        (i, j) = input_combo[n]
        node_time=pd.to_datetime(group.loc[n+1, 'time'])
        #print(node_time)

        tp_data = data.sel(time=node_time).mean(dim='time')

        tp_mean=np.mean(tp_data).values
        #tp_=clip_data-tp_mean

        gph_data= gph.sel(time=node_time).mean(dim='time')
        qu_data = qu.sel(time=node_time).mean(dim='time').values
        qv_data = qv.sel(time=node_time).mean(dim='time').values

        ua=qu_data[::interval,::interval]
        va=qv_data[::interval,::interval]

        magnitude = np.sqrt(ua**2 + va**2)
        magnitude_mean=np.mean(magnitude)


        levels = np.arange(-50, 51, 5)
        levels1 = [5860,5880]
        ax=axs[i,j]
        ax.annotate(f'#{n+1}', xy=(0.98, 0.84), **annotate_kwargs2)
        #ax.annotate(f'{tp_mean:.4f}', xy=(0.01, 0.06), **annotate_kwargs1)
        #ax.annotate(f'→ {magnitude_mean:.1f}', xy=(0.01, 0.06), **annotate_kwargs1)
        '''
        mk_result_per_p=feature['mk_result_per_p'][n]
        slope_per=feature['slope_per'][n]

        if slope_per<0:
            color='blue'
        else:
            color='red'
        
        if mk_result_per_p<0.01:
            text='**'
        elif mk_result_per_p<0.05:
            text='*'
        else:
            text=''
            
        ax.annotate(f'{slope_per*1000:.4f}', xy=(0.98, 0.04), xycoords='axes fraction',horizontalalignment='right',fontsize=6,color=color)
        ax.annotate(text, xy=(0.98, 0.14), xycoords='axes fraction',horizontalalignment='right',fontsize=9,color=color)
        '''

        norm = Normalize(vmin=0, vmax=600)

        ax.coastlines(linewidth=0.2, alpha=0.8)
        ax.add_geometries(shp.geometry, crs=ccrs.PlateCarree(), facecolor='none', edgecolor='black', linewidth=0.2, alpha=0.8)
        #title=r'$\mathbf{(a)}$ Dry-to-wet occurence frequency'
        #ax.set_extent([90, 112, 36, 24], crs=ccrs.PlateCarree())
        cf = ax.contourf(lons, lats, tp_data, transform=ccrs.PlateCarree(), levels=levels, extend='both', cmap="RdBu_r", norm = TwoSlopeNorm(vcenter=0))
        c = ax.contour(lons, lats, gph_data, transform=ccrs.PlateCarree(), levels=levels1, colors='k', linewidths=0.25, linestyles='dashed')
        ax.clabel(c, fontsize=2.5)
        quiver = ax.quiver(lons1, lats1, ua, va, magnitude, pivot='mid', scale=1800, width=0.002,  cmap='viridis', norm=norm, transform=ccrs.PlateCarree())

       
        for spine in ax.spines.values():
            spine.set_linewidth(0.2)

    fig.subplots_adjust(wspace=0.06, hspace=0.06)        
    plt.subplots_adjust(left=0.35, right=0.7, top=0.65, bottom=0.35) 


    ax1 = fig.add_axes([0.355, 0.335, 0.16, 0.06])
    ax1.spines['top'].set_visible(False)
    ax1.spines['bottom'].set_visible(False)
    ax1.spines['left'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.set_xticks([]) 
    ax1.set_yticks([]) 
    ax1.set_xticklabels([])  
    ax1.set_yticklabels([]) 
    ax1.set_facecolor('none')
    cbar = plt.colorbar(cf, ax=ax1, orientation='horizontal', aspect=40, pad=0.05)
    cbar.set_ticks(np.arange(-50, 51, 25))
    cbar.outline.set_linewidth(0.2)
    cbar.ax.tick_params(labelsize=5.5, length=2, width=0.2)

    ax2 = fig.add_axes([0.535, 0.335, 0.16, 0.06])
    ax2.spines['top'].set_visible(False)
    ax2.spines['bottom'].set_visible(False)
    ax2.spines['left'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.set_xticks([])  
    ax2.set_yticks([])  
    ax2.set_xticklabels([])  
    ax2.set_yticklabels([])  
    ax2.set_facecolor('none')
    cbar1 = plt.colorbar(quiver, ax=ax2, extend='max', orientation='horizontal', aspect=40, pad=0.05)
    cbar1.set_ticks(np.arange(0, 600+100, 100))
    cbar1.outline.set_linewidth(0.2)
    cbar1.ax.tick_params(labelsize=5.5, length=2, width=0.2)
    #fig.text(0.69,0.372,'m',size=7)


    fig.savefig(in_path+f"EA_gph_iwv_link_grid3x4_total.png",dpi=1000,bbox_inches='tight')
    plt.show()